In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

from utils import forward_selection, mdc_select, bcorsis_select, kfilter_select


    an issue that caused a segfault when used with rpy2:
    https://github.com/rstudio/reticulate/pull/1188
    Make sure that you use a version of that package that includes
    the fix.
    

## 1. Data

In [2]:
housing = fetch_california_housing()
X_real = housing.data
y      = housing.target

feature_names_real = list(housing.feature_names)
n, p_real = X_real.shape
print(f"California Housing: {n} samples, {p_real} real features")

rng_noise = np.random.default_rng(42)

# Type 1: Noisy copies (7 features) — real feature + small Gaussian noise
n_copies  = 7
copy_src  = rng_noise.choice(p_real, size=n_copies, replace=False)
X_copies  = X_real[:, copy_src] + 0.1 * rng_noise.standard_normal((n, n_copies))
copy_names = [f"NoisyCopy_{feature_names_real[j]}" for j in copy_src]

# Type 2: Random linear combinations (7 features)
n_lincomb  = 7
W          = rng_noise.standard_normal((p_real, n_lincomb))
W         /= np.linalg.norm(W, axis=0, keepdims=True)
X_lincomb  = X_real @ W + 0.1 * rng_noise.standard_normal((n, n_lincomb))
lincomb_names = [f"LinCombo_{i+1}" for i in range(n_lincomb)]

# Type 3: Pure i.i.d. noise (500 features)
n_pure   = 500
X_pure   = rng_noise.standard_normal((n, n_pure))
pure_names = [f"PureNoise_{i+1}" for i in range(n_pure)]

X_noise     = np.hstack([X_copies, X_lincomb, X_pure])
noise_names = copy_names + lincomb_names + pure_names
p_noise     = X_noise.shape[1]

X_all      = np.hstack([X_real, X_noise])
feat_names = feature_names_real + noise_names
p_total    = X_all.shape[1]

scaler = StandardScaler()
X_all  = scaler.fit_transform(X_all)

print(f"Noise features: {n_copies} noisy copies + {n_lincomb} linear combos + {n_pure} pure noise = {p_noise}")
print(f"Total features: {p_total}")

California Housing: 20640 samples, 8 real features
Noise features: 7 noisy copies + 7 linear combos + 500 pure noise = 514
Total features: 522


## 2. Train / Test Split

In [3]:
n_total = len(y)
half    = n_total // 2

rng_split = np.random.default_rng(0)
perm      = rng_split.permutation(n_total)

X_screen_all = X_all[perm[:half]]
y_screen_all = y[perm[:half]]

X_reg = X_all[perm[half:]]
y_reg = y[perm[half:]]

X_train, X_test, y_train, y_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=0
)

print(f"Screening half : {len(y_screen_all)} samples")
print(f"Regression half: {len(y_reg)} samples  (Train: {len(y_train)} | Test: {len(y_test)})")

Screening half : 10320 samples
Regression half: 10320 samples  (Train: 8256 | Test: 2064)


## 3. Screening

In [4]:
N_SCREEN = 2000
K_NN     = 10

rng = np.random.default_rng(1)
if len(y_screen_all) > N_SCREEN:
    sc_idx = rng.choice(len(y_screen_all), size=N_SCREEN, replace=False)
    X_sc = X_screen_all[sc_idx]
    y_sc = y_screen_all[sc_idx]
else:
    X_sc = X_screen_all
    y_sc = y_screen_all

print("Running NNCMI (k=10) forward selection...", flush=True)
nncmi_feats_k10 = forward_selection(X_sc, y_sc, k_nn=10, statistic="nncmi")

print("Running NNCMI (k=25) forward selection...", flush=True)
nncmi_feats_k25 = forward_selection(X_sc, y_sc, k_nn=25, statistic="nncmi")

print("Running Chatterjee (kfoci) forward selection...", flush=True)
chatt_feats = forward_selection(X_sc, y_sc, k_nn=K_NN, statistic="chatterjee")

print("Running MDC screening (own stopping)...", flush=True)
mdc_feats = mdc_select(X_sc, y_sc)

m = max(len(nncmi_feats_k10), len(nncmi_feats_k25), len(chatt_feats))
print(f"Running MDC screening (matched m={m})...", flush=True)
mdc_feats_matched = mdc_select(X_sc, y_sc, nsis=m)

print("Running BcorSIS screening...", flush=True)
bcorsis_feats = bcorsis_select(X_sc, y_sc, nsis=m)

print("Running Kfilter screening...", flush=True)
kfilter_feats = kfilter_select(X_sc, y_sc, nsis=m)

print("Done.", flush=True)

Running NNCMI (k=10) forward selection...
Running NNCMI (k=25) forward selection...
Running Chatterjee (kfoci) forward selection...
Running MDC screening (own stopping)...
Running MDC screening (matched m=6)...
Running BcorSIS screening...
Running Kfilter screening...
Done.


## 4. Selected Features

In [5]:
def noise_type(idx, p_real, n_copies, n_lincomb):
    if idx < p_real:
        return "Real"
    offset = idx - p_real
    if offset < n_copies:
        return "Noise:Copy"
    elif offset < n_copies + n_lincomb:
        return "Noise:LinCombo"
    else:
        return "Noise:Pure"


def summarise_selection(label, indices, feat_names, p_real, n_copies, n_lincomb):
    rows = []
    for rank, idx in enumerate(indices, start=1):
        rows.append({
            "Rank":    rank,
            "Index":   idx,
            "Feature": feat_names[idx],
            "Type":    noise_type(idx, p_real, n_copies, n_lincomb),
        })
    print(f"\n{label} — {len(indices)} features selected:")
    display(pd.DataFrame(rows))


summarise_selection("NNCMI (k=10)",  nncmi_feats_k10, feat_names, p_real, n_copies, n_lincomb)
summarise_selection("NNCMI (k=25)",  nncmi_feats_k25, feat_names, p_real, n_copies, n_lincomb)
summarise_selection("Chatterjee (kfoci)", chatt_feats, feat_names, p_real, n_copies, n_lincomb)
summarise_selection(
    f"MDC (own, nsis=⌊{N_SCREEN}/log({N_SCREEN})⌋={int(N_SCREEN/np.log(N_SCREEN))})",
    sorted(mdc_feats), feat_names, p_real, n_copies, n_lincomb
)
summarise_selection(f"MDC (matched, m={m})", sorted(mdc_feats_matched), feat_names, p_real, n_copies, n_lincomb)
summarise_selection("BcorSIS", sorted(bcorsis_feats), feat_names, p_real, n_copies, n_lincomb)
summarise_selection("Kfilter", sorted(kfilter_feats), feat_names, p_real, n_copies, n_lincomb)


NNCMI (k=10) — 4 features selected:


,Rank,Index,Feature,Type
0,1,0,MedInc,Real
1,2,7,Longitude,Real
2,3,6,Latitude,Real
3,4,14,NoisyCopy_AveOccup,Noise:Copy



NNCMI (k=25) — 6 features selected:


,Rank,Index,Feature,Type
0,1,0,MedInc,Real
1,2,7,Longitude,Real
2,3,6,Latitude,Real
3,4,5,AveOccup,Real
4,5,3,AveBedrms,Real
5,6,14,NoisyCopy_AveOccup,Noise:Copy



Chatterjee (kfoci) — 5 features selected:


...

,Rank,Index,Feature,Type
0,1,0,MedInc,Real
1,2,2,AveRooms,Real
2,3,5,AveOccup,Real
3,4,12,NoisyCopy_MedInc,Noise:Copy
4,5,13,NoisyCopy_AveRooms,Noise:Copy
5,6,14,NoisyCopy_AveOccup,Noise:Copy



BcorSIS — 6 features selected:


,Rank,Index,Feature,Type
0,1,0,MedInc,Real
1,2,2,AveRooms,Real
2,3,5,AveOccup,Real
3,4,12,NoisyCopy_MedInc,Noise:Copy
4,5,13,NoisyCopy_AveRooms,Noise:Copy
5,6,14,NoisyCopy_AveOccup,Noise:Copy



Kfilter — 6 features selected:


,Rank,Index,Feature,Type
0,1,1,HouseAge,Real
1,2,3,AveBedrms,Real
2,3,7,Longitude,Real
3,4,10,NoisyCopy_Population,Noise:Copy
4,5,11,NoisyCopy_AveBedrms,Noise:Copy
5,6,14,NoisyCopy_AveOccup,Noise:Copy


## 5. XGBoost Evaluation

In [6]:
def fit_and_eval(X_train, X_test, y_train, y_test, feature_indices):
    model = XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=5,
        subsample=0.8, colsample_bytree=0.8,
        random_state=0, n_jobs=-1, verbosity=0,
    )
    model.fit(
        X_train[:, feature_indices], y_train,
        eval_set=[(X_test[:, feature_indices], y_test)],
        verbose=False,
    )
    y_pred = model.predict(X_test[:, feature_indices])
    mse  = mean_squared_error(y_test, y_pred)
    return mse, np.sqrt(mse)


scenarios = {
    "NNCMI (k=10)":       nncmi_feats_k10,
    "NNCMI (k=25)":       nncmi_feats_k25,
    "Chatterjee (kfoci)": chatt_feats,
    "BcorSIS":            bcorsis_feats,
    "Kfilter":            kfilter_feats,
    "MDC (matched m)":    mdc_feats_matched,
    "MDC (own)":          mdc_feats,
    "All features":       list(range(p_total)),
    "Oracle (real only)": list(range(p_real)),
}

rows = []
for name, feats in scenarios.items():
    print(f"Fitting {name}...", flush=True)
    mse, rmse = fit_and_eval(X_train, X_test, y_train, y_test, feats)
    n_real_sel = sum(1 for f in feats if f < p_real)
    rows.append({
        "Method":           name,
        "# Selected":       len(feats),
        "# Real Selected":  n_real_sel,
        "# Noise Selected": len(feats) - n_real_sel,
        "Test MSE":         round(mse,  4),
        "Test RMSE":        round(rmse, 4),
    })

results_df = pd.DataFrame(rows).set_index("Method")
display(results_df)

Fitting NNCMI (k=10)...
Fitting NNCMI (k=25)...
Fitting Chatterjee (kfoci)...
Fitting BcorSIS...
Fitting Kfilter...
Fitting MDC (matched m)...
Fitting MDC (own)...
Fitting All features...
Fitting Oracle (real only)...


,# Selected,# Real Selected,# Noise Selected,Test MSE,Test RMSE
Method,,,,,
NNCMI (k=10),4,3,1,0.2206,0.4697
NNCMI (k=25),6,5,1,0.2292,0.4788
Chatterjee (kfoci),5,4,1,0.2246,0.4739
BcorSIS,6,3,3,0.4958,0.7042
Kfilter,6,3,3,0.7457,0.8636
MDC (matched m),6,3,3,0.4972,0.7052
MDC (own),263,8,255,0.2533,0.5033
All features,522,8,514,0.2558,0.5057
Oracle (real only),8,8,0,0.2036,0.4512


## 6. LaTeX Table

In [7]:
latex = results_df.to_latex(column_format="|l|c|c|c|c|c|", escape=False)
latex = (
    latex
    .replace("\\toprule",    "\\hline")
    .replace("\\midrule",    "\\hline")
    .replace("\\bottomrule", "\\hline")
)
print(latex)

\begin{tabular}{|l|c|c|c|c|c|}
\hline
 & # Selected & # Real Selected & # Noise Selected & Test MSE & Test RMSE \\
Method &  &  &  &  &  \\
\hline
NNCMI (k=10) & 4 & 3 & 1 & 0.220600 & 0.469700 \\
NNCMI (k=25) & 6 & 5 & 1 & 0.229200 & 0.478800 \\
Chatterjee (kfoci) & 5 & 4 & 1 & 0.224600 & 0.473900 \\
BcorSIS & 6 & 3 & 3 & 0.495800 & 0.704200 \\
Kfilter & 6 & 3 & 3 & 0.745700 & 0.863600 \\
MDC (matched m) & 6 & 3 & 3 & 0.497200 & 0.705200 \\
MDC (own) & 263 & 8 & 255 & 0.253300 & 0.503300 \\
All features & 522 & 8 & 514 & 0.255800 & 0.505700 \\
Oracle (real only) & 8 & 8 & 0 & 0.203600 & 0.451200 \\
\hline
\end{tabular}

